# Silver: NOTAMs
Parses JSON + extracts structured fields from the NOTAM `condition` string, including the ICAO Q-code which encodes the nature/severity of the notice.

## Imports and Setup

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:
import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    BRONZE_NOTAMS_PATH,
    abfss, SILVER_CONTAINER,
    configure_storage_auth,
)

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
)

SILVER_NOTAMS_PATH = abfss(SILVER_CONTAINER, "notams")

## Read from Bronze

In [0]:
configure_storage_auth(spark)

In [0]:
bronze = spark.read.format("delta").load(BRONZE_NOTAMS_PATH)

## Enforce Schema on JSON

In [0]:
notam_payload = StructType([
    StructField("location", StringType()),
    StructField("number", StringType()),
    StructField("class", StringType()),
    StructField("startdateutc", StringType()),
    StructField("enddateutc", StringType()),
    StructField("condition", StringType()),
])

message_schema = StructType([
    StructField("ingestion_ts_utc", StringType()),
    StructField("queried_airport", StringType()),
    StructField("chunk_from", StringType()),
    StructField("chunk_to", StringType()),
    StructField("source", StringType()),
    StructField("source_endpoint", StringType()),
    StructField("payload", notam_payload),
])

## Parse JSON

In [0]:
parsed = (
    bronze
    .select(
        "kafka_key", "kafka_timestamp", "ingestion_ts_utc",
        F.from_json("kafka_value", message_schema).alias("msg")
    )
    .select(
        "kafka_key", "kafka_timestamp",
        F.col("ingestion_ts_utc").alias("bronze_ingestion_ts_utc"),
        "msg.*",
    )
)


## Parse Timestamps
Sample format: `2026-04-11t06:00:00`

In [0]:
parsed_ts = (
    parsed
    .withColumn(
        "start_ts_utc",
        F.to_timestamp(F.upper(F.col("payload.startdateutc")), "yyyy-MM-dd'T'HH:mm:ss")
    )
    .withColumn(
        "end_ts_utc",
        F.to_timestamp(F.upper(F.col("payload.enddateutc")), "yyyy-MM-dd'T'HH:mm:ss")
    )
)

## Extract Q-Code from the Condition string
This is one of the most crucial parts of my DE part when it comes to NOTAMs.

Q-codes follow the NOTAM standard: `q<FIR><subject><traffic><purpose><scope><lower><upper>`
 
We only care about the **2nd-5th characters** of the Q-code (subject + scope), like `qwplw` => subject=`wp`, scope=`lw`.
 
Examples:
- `qwplw` → parachuting (WP) + airspace (LW) => medium impact
- `qrtca` → restricted area (RT) + airspace area (CA) =? high impact
- `qmrlc` → runway (MR) + closed (LC) => **critical - runway closed!**

In [0]:
# Regex extracts the Q-code like `q<letter>{4}` like `qwplw`, `qmrlc`
q_code_regex = r"q([a-z]{4})"

silver = (
    parsed_ts
    .select(
        # core notam fields
        F.upper(F.col("payload.location")).alias("notam_location"),
        F.upper(F.col("queried_airport")).alias("queried_airport"),
        F.upper(F.col("payload.number")).alias("notam_number"),
        F.lower(F.col("payload.class")).alias("notam_class"),
        F.col("start_ts_utc"),
        F.col("end_ts_utc"),

        # q-code extraction
        F.regexp_extract(F.col("payload.condition"), q_code_regex, 1).alias("q_code"),

        # raw condition text preserved 
        F.col("payload.condition").alias("condition_text"),

        # metadata
        F.col("bronze_ingestion_ts_utc"),
        F.col("kafka_timestamp"),
        F.to_date("chunk_from").alias("chunk_from"),
        F.to_date("chunk_to").alias("chunk_to"),
    )
    # decode Q-code into human-readable + severity
    .withColumn("q_subject", F.substring(F.col("q_code"), 1, 2))
    .withColumn("q_scope", F.substring(F.col("q_code"), 3, 2))
)


## Decode Q-code severity

The first 2 chars (subject) are the most important. We map common codes to a severity scale `{low, medium, high, critical}` for feature engineering.

A subset of the most impactful Q-code subjects for delays.  
Full reference: ICAO Doc 8126 Annex

In [0]:
severity_mapping = {
    # Critical - runway/airport essentially closed
    "MR": "critical",  # runway
    "MX": "critical",  # taxiway (major)
    "FA": "critical",  # aerodrome (facility)

    # High - significant operational impact
    "RT": "high",      # temporarily restricted area
    "RR": "high",      # restricted area
    "RP": "high",      # prohibited area
    "RD": "high",      # danger area
    "LC": "high",      # closed
    "LT": "high",      # limited
    "WP": "high",      # parachuting
    "WB": "high",      # balloon/unmanned

    # Medium - advisories
    "WT": "medium",    # test/training
    "WC": "medium",    # aerial display
    "IL": "medium",    # ILS
    "IC": "medium",    # ILS glide path
    "NN": "medium",    # navigation warning
    "CA": "medium",    # ATS route
    "CR": "medium",    # ATS route requiring modification

    # Low - informational
    "PA": "low",       # standard instrument arrival
    "PD": "low",       # standard instrument departure
    "PI": "low",       # instrument approach procedure
}

# Build a CASE WHEN ... expression
severity_expr = F.lit("unknown")
for subject, sev in severity_mapping.items():
    severity_expr = F.when(F.col("q_subject") == subject.lower(), sev).otherwise(severity_expr)

silver_with_severity = silver.withColumn("severity", severity_expr)


## Quality Filters

In [0]:
silver_filtered = (
    silver_with_severity
    .filter(F.col("start_ts_utc").isNotNull())
    .filter(F.col("end_ts_utc").isNotNull())
    .filter(F.col("end_ts_utc") > F.col("start_ts_utc"))  # sane time ranges
)


## Deduplicate
Same NOTAM re-ingested if the notam_number if the same

In [0]:
from pyspark.sql.window import Window

w = (
    Window
    .partitionBy("notam_number", "notam_location")
    .orderBy(F.col("bronze_ingestion_ts_utc").desc())
)

silver_deduped = (
    silver_filtered
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
    .withColumn("silver_processed_ts_utc", F.current_timestamp())
)

## Write to Silver

In [0]:
(
    silver_deduped.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_NOTAMS_PATH)
)

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver.notams
    USING DELTA
    LOCATION '{SILVER_NOTAMS_PATH}'
""")

## Exit

In [0]:
dbutils.notebook.exit(f'{{"rows_in_silver": {silver_deduped.count()}}}')